In [ ]:
from google.colab import drive
drive.mount('/gdrive',force_remount=True)

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle

In [ ]:
!cp kaggle.json ~/.kaggle/

In [ ]:
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# === KITSUNE — Kaggle + unificación/limpieza + taxonomía final (estilo UNSW)
# VERSIÓN CORREGIDA respecto a la original de TFM1:
#   1. Se recupera "Reconnaissance" como clase propia (antes se
#      fusionaba con "PortScan" en MAP_ATTACK, igual que ocurría en el
#      script de UNSW-NB15).
#   2. NOTA IMPORTANTE sobre IP: a diferencia de UNSW-NB15 y CICIDS2017,
#      el dataset Kitsune NO tiene columnas de IP de origen/destino en
#      ninguno de sus CSV crudos -- son estadísticas derivadas paquete a
#      paquete (jitter, tamaños, tiempos entre paquetes...), no flujos
#      con IP. Esto es una limitación estructural del propio dataset,
#      no una decisión de limpieza que se pueda "deshacer": no se
#      fabrica ninguna IP falsa aquí para no contaminar el fichero
#      "_full_clean" (evidencia real) con datos inventados. Si se
#      necesitan escenarios de agregación por origen para Kitsune, hay
#      que construirlos por separado, asignando IP sintéticas SOLO en
#      la parte de generación sintética, nunca en el real.
# Salida: /content/KITSUNE_full_clean_v2.csv con columnas al inicio: attack_cat, label
!pip install -q kaggle pandas numpy tqdm

import os, sys, glob, time, shutil, zipfile, subprocess, gc
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ============================================================
# 1) TAXONOMÍA FINAL (COMÚN A UNSW / CICIDS / KITSUNE)
# ============================================================
ALLOWED_ATTACKS = {
    "Normal",
    "Fuzzers","Exploits","DoS","Reconnaissance","Generic","Analysis",
    "Shellcode","Backdoors","DDoS","PortScan","MitM","BruteForce","Worms"
}

MAP_ATTACK = {
    # Normal
    "benign":"Normal","Benign":"Normal","BENIGN":"Normal",
    "normal":"Normal","Normal":"Normal",
    "nan":"Normal","NaN":"Normal","None":"Normal","":"Normal",

    # clave:
    "Attack": "Generic","attack": "Generic","Unknown": "Generic","unknown":"Generic",

    # DoS / DDoS
    "dos":"DoS","Dos":"DoS","DoS":"DoS",
    "ddos":"DDoS","Ddos":"DDoS","DDoS":"DDoS",

    # PortScan (Reconnaissance ya NO se fusiona aquí: queda como clase
    # propia, igual que en UNSW-NB15 y en la taxonomía de MODEXRE)
    "Port Scan":"PortScan","port scan":"PortScan",
    "portscan":"PortScan","Portscan":"PortScan","PortScan":"PortScan",
    "Reconnaissance":"Reconnaissance","reconnaissance":"Reconnaissance",

    # MITM
    "mitm":"MitM","MITM":"MitM","MitM":"MitM",

    # BruteForce
    "Bruteforce":"BruteForce","bruteforce":"BruteForce","Brute Force":"BruteForce","brute force":"BruteForce",

    # Worms
    "worms":"Worms","Worms":"Worms","worm":"Worms","Worm":"Worms",
}
def normalize_attack_cat_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.replace(MAP_ATTACK)
    s = s.apply(lambda x: "Normal" if str(x).strip().lower() == "normal" else x)
    # Vacío -> Normal. Cualquier OTRA clase se conserva tal cual, esté
    # o no en ALLOWED_ATTACKS: no se colapsa a "Generic".
    s = s.apply(lambda x: "Normal" if str(x).strip() == "" else str(x).strip())
    return s

def enforce_attackcat_label(df: pd.DataFrame) -> pd.DataFrame:
    """
    CHECK DURO DEFINITIVO:
      - attack_cat == Normal  => label = 0
      - attack_cat != Normal  => label = 1
    """
    df = df.copy()
    if "attack_cat" not in df.columns:
        raise ValueError("Falta 'attack_cat'.")
    df["attack_cat"] = normalize_attack_cat_series(df["attack_cat"])
    df["label"] = (df["attack_cat"] != "Normal").astype(int).astype("category")
    return df

# ============================================================
# 2) UTILIDADES KAGGLE
# ============================================================
def run(cmd):
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return p.returncode, p.stdout, p.stderr

def ensure_kaggle_token():
    run([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
    kag_dir = Path.home() / ".kaggle"
    kag_dir.mkdir(parents=True, exist_ok=True)
    token = kag_dir / "kaggle.json"
    if not token.exists():
        from google.colab import files  # type: ignore
        print("→ Sube tu kaggle.json (Kaggle > Account > Create API Token)…")
        up = files.upload()
        cand = [k for k in up.keys() if k.endswith("kaggle.json")]
        if not cand:
            raise FileNotFoundError("No se subió kaggle.json.")
        shutil.copy(cand[0], token)
    os.chmod(token, 0o600)

def robust_read_csv(path):
    for enc in ("utf-8","latin1","cp1252"):
        try:
            return pd.read_csv(path, low_memory=False, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path, low_memory=False, engine="python")

def normalize_cols(df):
    df.columns = [str(c).strip().replace(" ","_").replace("/","_").replace("-","_").lower() for c in df.columns]
    return df

# ============================================================
# 3) DESCARGA KITSUNE (CANDIDATOS)
# ============================================================
ensure_kaggle_token()

CANDIDATES = [
    "ernie55ernie/5-tuple-packets-of-kitsune-network-attack-dataset",  # ✅ el que te ha funcionado
    "ymirsky/network-attack-dataset-kitsune",
    "ycshin/kitsune-network-attack-dataset",
]

zip_path = None
for ds in CANDIDATES:
    print(f"\n[INFO] Intentando Kaggle: {ds}")
    pre = set(glob.glob("/content/*.zip"))
    rc, out, err = run(["kaggle","datasets","download","-d",ds,"-p","/content","-w"])
    if rc == 0:
        time.sleep(1.0)
        post = set(glob.glob("/content/*.zip"))
        newz = list(post - pre)
        zip_path = newz[0] if newz else (max(list(post), key=os.path.getmtime) if post else None)
        if zip_path and os.path.exists(zip_path):
            print("[OK] ZIP descargado:", zip_path)
            break
    print("[FALLO]", (err or out)[0:200].replace("\n"," "), "…")

if not zip_path:
    raise RuntimeError("No se pudo descargar Kitsune con los candidatos. Sube el ZIP manualmente.")

TARGET_DIR = "/content/kitsune"
os.makedirs(TARGET_DIR, exist_ok=True)
print(f"[INFO] Descomprimiendo {zip_path} → {TARGET_DIR}")
with zipfile.ZipFile(zip_path) as z:
    z.extractall(TARGET_DIR)

# ============================================================
# 4) INFERIR attack_cat DESDE NOMBRE DE FICHERO (KITSUNE)
# ============================================================
def attack_cat_from_path(path: str):
    s = os.path.basename(path).lower()

    # benign/baseline si existiera
    if any(t in s for t in ["benign","normal","baseline","idle","clean"]):
        return "Normal"

    # kitsune ataques típicos
    if "mirai" in s or "botnet" in s:
        return "DDoS"
    if "scan" in s:
        return "Reconnaissance"
    if "fuzz" in s:
        return "Fuzzers"
    if "dos" in s or "flood" in s or "reneg" in s:
        return "DoS"
    if "mitm" in s or "arp" in s or "spoof" in s:
        return "MitM"

    # Si el nombre de fichero no encaja con ningún patrón conocido, se
    # CONSERVA como clase propia derivada del propio nombre de fichero
    # (sin extensión, title-case), en vez de forzarlo a "Generic". Así
    # un ataque de Kitsune no reconocido de antemano (p.ej. un fichero
    # nuevo añadido al dataset) mantiene su propia identidad.
    stem = os.path.splitext(os.path.basename(path))[0]
    return " ".join(stem.replace("_", " ").replace("-", " ").split()).title() or "Generic"

# ============================================================
# 5) CARGA + UNIFICACIÓN (CON PROGRESO)
# ============================================================
csv_files = sorted(glob.glob(f"{TARGET_DIR}/**/*.csv", recursive=True))
if not csv_files:
    raise FileNotFoundError("No se encontraron CSV dentro del ZIP extraído.")

print(f"[INFO] CSV encontrados: {len(csv_files)}")

dfs = []
cats = []

for f in tqdm(csv_files, desc="Cargando CSV Kitsune", unit="csv"):
    df_i = robust_read_csv(f)
    df_i = normalize_cols(df_i)

    file_cat = attack_cat_from_path(f)

    # si el csv trae alguna etiqueta interna, la usamos SOLO para detectar benign/ataque
    label_cols = [c for c in df_i.columns if c in ("label","class","attack","anomaly","is_anomaly","malicious")]
    if label_cols:
        lab = df_i[label_cols[0]].astype(str).str.strip().str.lower()
        df_i = df_i.drop(columns=label_cols, errors="ignore")
        is_benign = lab.isin({"0","benign","normal","false","no"})
        cat = np.where(is_benign, "Normal", file_cat)
    else:
        cat = np.array([file_cat] * len(df_i), dtype=object)

    dfs.append(df_i)
    cats.append(cat)

df = pd.concat(dfs, ignore_index=True)
df = normalize_cols(df)

attack_cat_all = np.concatenate(cats, axis=0)
df["attack_cat"] = pd.Series(attack_cat_all, dtype="object")

del dfs, cats
gc.collect()

# ============================================================
# 6) CHECK DURO + LIMPIEZA NUMÉRICA
# ============================================================
df = enforce_attackcat_label(df)

# Limpieza numérica mínima (sin dropna global)
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns.tolist() if c not in ("label",)]
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)

for c in tqdm(num_cols, desc="Imputando numéricas (mediana)", unit="col"):
    if df[c].isna().any():
        med = df[c].median()
        df[c] = df[c].fillna(0.0 if np.isnan(med) else med)

# Drop columnas constantes/vacías
const_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
df = df.drop(columns=const_cols, errors="ignore")

# ============================================================
# 7) REORDENAR COLUMNAS: attack_cat, label PRIMERO
# ============================================================
cols = df.columns.tolist()
cols = ["attack_cat", "label"] + [c for c in cols if c not in ("attack_cat","label")]
df = df[cols]

# ============================================================
# 8) GUARDAR + VERIFICAR
# ============================================================
OUT = "/content/KITSUNE_full_clean_v2.csv"
df.to_csv(OUT, index=False, encoding="utf-8")

print("\n[OK] Guardado:", OUT)
print("Shape:", df.shape)

print("\nlabel:")
print(df["label"].astype(str).value_counts(normalize=True).round(3))

print("\nattack_cat (todas las clases, incluye Reconnaissance y Worms si están presentes):")
print(df["attack_cat"].value_counts())
print("\nReconnaissance detectada como clase propia →", int((df["attack_cat"]=="Reconnaissance").sum()), "filas")
print("Worms detectada como clase propia →", int((df["attack_cat"]=="Worms").sum()), "filas")
print("\n[NOTA] Kitsune no tiene columnas de IP de origen/destino en el dataset "
      "crudo (son estadísticas derivadas paquete a paquete, no flujos con IP). "
      "No se fabrica ninguna IP en este fichero: es evidencia real, no debe "
      "contener datos inventados.")

print("\n[CHECK FINAL]")
print("Generic con label=0 →", int(((df.attack_cat=="Generic")&(df.label.astype(int)==0)).sum()))
print("Normal con label=1  →", int(((df.attack_cat=="Normal")&(df.label.astype(int)==1)).sum()))

In [ ]:
# === KITSUNE — GaussianCopula "1 sintetizador por clase" (REAL condicional) + anti-OOM + progreso ===
!pip install -q sdv packaging tqdm

import os, gc, time, math
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import Counter
from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer

# ===================== CONFIG =====================
REAL_CSV  = "/content/KITSUNE_full_clean_v2.csv"
SYN_CSV   = "/content/synthetic_kitsune_ctgan_v2.csv"
META_DIR  = "/content/kitsune_metadata_per_class_v2"   # guardará un json por clase (opcional pero útil)

RANDOM_STATE = 42

# Cuotas sintéticas finales
NORMAL_N      = 200_000
ATTACK_N_EACH = 30_000

# Lectura por chunks (Kitsune es enorme)
CHUNK_SIZE = 300_000

# Muestras para entrenar SDV por clase (sube/baja según RAM)
TRAIN_NORMAL_N = 120_000     # muestra para entrenar "Normal"
TRAIN_ATTACK_N = 30_000      # muestra para entrenar cada ataque (DoS/MitM/...)

# SDV params
DEFAULT_DISTRIBUTION = "gamma"
ENFORCE_MINMAX = False
# ================================================

os.makedirs(META_DIR, exist_ok=True)
rng = np.random.RandomState(RANDOM_STATE)

def normalize_cols(cols):
    return [str(c).strip().replace(" ", "_").replace("/", "_").replace("-", "_").lower() for c in cols]

def enforce_attackcat_label_inplace(df_any: pd.DataFrame) -> pd.DataFrame:
    if "attack_cat" not in df_any.columns:
        raise ValueError("Falta 'attack_cat' en el CSV.")
    df_any["attack_cat"] = df_any["attack_cat"].astype(str).str.strip()
    # CHECK DURO: label depende SOLO de attack_cat
    df_any["label"] = (df_any["attack_cat"].str.lower() != "normal").astype(int)
    return df_any

def approx_line_count(path: str) -> int:
    with open(path, "rb") as f:
        return max(sum(1 for _ in f) - 1, 0)

def append_csv(df_part: pd.DataFrame, path: str):
    header = not os.path.exists(path)
    df_part.to_csv(path, index=False, mode="a", header=header)

def reorder_attackcat_label_first(df_part: pd.DataFrame) -> pd.DataFrame:
    # garantiza attack_cat, label al principio
    cols = df_part.columns.tolist()
    front = [c for c in ["attack_cat", "label"] if c in cols]
    rest = [c for c in cols if c not in front]
    return df_part[front + rest]

def build_class_sample(klass: str, target_n: int, n_chunks: int) -> pd.DataFrame:
    """
    Construye una muestra SOLO de la clase 'klass' leyendo el CSV por chunks.
    """
    parts = []
    seen = 0

    reader = pd.read_csv(REAL_CSV, low_memory=False, chunksize=CHUNK_SIZE)
    for ch in tqdm(reader, total=n_chunks, desc=f"Sampling REAL [{klass}]", unit="chunk"):
        ch.columns = normalize_cols(ch.columns.tolist())
        ch = enforce_attackcat_label_inplace(ch)

        if klass.lower() == "normal":
            sub = ch[ch["attack_cat"].str.lower() == "normal"]
        else:
            sub = ch[ch["attack_cat"] == klass]

        if len(sub) > 0:
            need = target_n - seen
            take = min(need, len(sub))
            # sample aleatorio dentro del chunk (si el chunk es enorme, evita sesgo por orden)
            samp = sub.sample(n=take, random_state=int(rng.randint(0, 1e9)))
            parts.append(samp)
            seen += len(samp)

        del ch, sub
        gc.collect()

        if seen >= target_n:
            break

    if not parts:
        return pd.DataFrame()

    df_sample = pd.concat(parts, ignore_index=True)
    del parts
    gc.collect()

    # barajar
    df_sample = df_sample.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

    # asegurar tamaño exacto si sobra
    if len(df_sample) > target_n:
        df_sample = df_sample.sample(n=target_n, random_state=RANDOM_STATE).reset_index(drop=True)

    return df_sample

def fit_synth_for_class(df_train: pd.DataFrame, klass: str) -> GaussianCopulaSynthesizer:
    """
    Entrena un GaussianCopulaSynthesizer SOLO con df_train (una clase).
    """
    # SDV no admite category
    df_sdv = df_train.copy()
    for c in df_sdv.columns:
        if str(df_sdv[c].dtype) == "category":
            df_sdv[c] = df_sdv[c].astype("object")

    # forzar strings en categóricas clave
    df_sdv["attack_cat"] = df_sdv["attack_cat"].astype(str)
    df_sdv["label"] = df_sdv["label"].astype(str)

    metadata = Metadata.detect_from_dataframe(df_sdv)
    meta_path = os.path.join(META_DIR, f"metadata_{klass}.json".replace(" ", "_"))
    metadata.save_to_json(meta_path)

    synth = GaussianCopulaSynthesizer(
        metadata,
        default_distribution=DEFAULT_DISTRIBUTION,
        enforce_min_max_values=ENFORCE_MINMAX
    )

    t0 = time.time()
    synth.fit(df_sdv)
    print(f"[OK] fit({klass}) en {(time.time()-t0)/60:.2f} min | train={df_sdv.shape}")

    del df_sdv
    gc.collect()

    return synth

# ===================== 0) Preparación: contar clases reales por chunks =====================
n_lines = approx_line_count(REAL_CSV)
n_chunks = max(1, int(math.ceil(n_lines / CHUNK_SIZE)))
print(f"[INFO] REAL filas aprox: {n_lines:,} | chunks: {n_chunks} | chunksize: {CHUNK_SIZE:,}")

cnt = Counter()
reader = pd.read_csv(REAL_CSV, low_memory=False, chunksize=CHUNK_SIZE)
for ch in tqdm(reader, total=n_chunks, desc="Scan clases REAL", unit="chunk"):
    ch.columns = normalize_cols(ch.columns.tolist())
    if "attack_cat" not in ch.columns:
        raise ValueError("El CSV REAL no tiene attack_cat.")
    cnt.update(ch["attack_cat"].astype(str).str.strip().tolist())
    del ch
    gc.collect()

attack_cats = sorted([c for c in cnt.keys() if str(c).strip().lower() != "normal"])
print("[INFO] attack_cats detectadas (sin Normal):", attack_cats)
print("[INFO] top10 real:", dict(Counter(cnt).most_common(10)))

if not attack_cats:
    raise RuntimeError("No se detectaron ataques en KITSUNE_full_clean.csv (solo Normal).")

# Acumulador para exportar, al final, una muestra de entrenamiento
# manejable (equivalente al df_train de los notebooks de UNSW-NB15 y
# CICIDS2017): aquí se entrena un sintetizador por clase y cada
# df_train_X se borra tras usarlo, así que se guarda una copia ligera
# de cada una antes de borrarla.
_train_sample_parts = []

# ===================== 1) Preparar salida =====================
if os.path.exists(SYN_CSV):
    os.remove(SYN_CSV)
print("[INFO] SYN_CSV:", SYN_CSV)

# ===================== 2) Generar Normal (sintetizador entrenado SOLO con Normal) =====================
print("\n==================== CLASE: Normal ====================")
df_train_N = build_class_sample("Normal", TRAIN_NORMAL_N, n_chunks=n_chunks)
if df_train_N.empty:
    raise RuntimeError("No pude construir muestra para Normal. Revisa attack_cat en el REAL.")
print("[INFO] train Normal:", df_train_N.shape, "| attack_cat:", df_train_N["attack_cat"].value_counts().to_dict())

synth_N = fit_synth_for_class(df_train_N, "Normal")

t0 = time.time()
synN = synth_N.sample(num_rows=NORMAL_N)
synN.columns = normalize_cols(synN.columns.tolist())
synN = enforce_attackcat_label_inplace(synN)

# forzar coherencia estricta
synN["attack_cat"] = "Normal"
synN["label"] = 0

synN = reorder_attackcat_label_first(synN)
append_csv(synN, SYN_CSV)

_train_sample_parts.append(df_train_N.copy())

del df_train_N, synth_N, synN
gc.collect()
print(f"[OK] Normal generado={NORMAL_N:,} en {(time.time()-t0)/60:.2f} min")

# ===================== 3) Generar ataques (1 sintetizador por ataque) =====================
print("\n==================== ATAQUES (por clase) ====================")

for cat in tqdm(attack_cats, desc="Ataques: entrenar+generar", unit="clase"):
    print(f"\n----- CLASE: {cat} -----")
    df_train_A = build_class_sample(cat, TRAIN_ATTACK_N, n_chunks=n_chunks)

    if df_train_A.empty:
        print(f"[AVISO] No pude construir muestra para {cat}. Se omite.")
        continue

    print("[INFO] train:", df_train_A.shape, "| dist:", df_train_A["attack_cat"].value_counts().to_dict())

    synth_A = fit_synth_for_class(df_train_A, cat)

    t1 = time.time()
    synA = synth_A.sample(num_rows=ATTACK_N_EACH)
    synA.columns = normalize_cols(synA.columns.tolist())
    synA = enforce_attackcat_label_inplace(synA)

    # forzar clase objetivo
    synA["attack_cat"] = str(cat)
    synA["label"] = 1

    synA = reorder_attackcat_label_first(synA)
    append_csv(synA, SYN_CSV)

    _train_sample_parts.append(df_train_A.copy())

    del df_train_A, synth_A, synA
    gc.collect()
    print(f"[OK] {cat} generado={ATTACK_N_EACH:,} en {(time.time()-t1)/60:.2f} min")

print("\n[OK] Sintético multiclase guardado en:", SYN_CSV)

# ===================== Exportar muestra manejable para MODEXRE =====================
# Unión de las muestras reales usadas para entrenar cada sintetizador
# por clase: es la que hay que subir a la pestaña Laboratorio de
# MODEXRE (ver misma nota en los notebooks de UNSW-NB15/CICIDS2017
# sobre el límite de subida de Streamlit y el CSV completo).
TRAIN_SAMPLE_CSV = "/content/KITSUNE_train_sample_v2.csv"
df_train_sample = pd.concat(_train_sample_parts, ignore_index=True)
df_train_sample = df_train_sample.sample(frac=1.0, random_state=42).reset_index(drop=True)
df_train_sample.to_csv(TRAIN_SAMPLE_CSV, index=False, encoding="utf-8")
print(f"[OK] Muestra de entrenamiento exportada → {TRAIN_SAMPLE_CSV}")
print(f"     Shape: {df_train_sample.shape}")
print(f"     Peso aproximado: {os.path.getsize(TRAIN_SAMPLE_CSV) / (1024*1024):.1f} MB")
print(f"     Distribución de attack_cat:")
print(df_train_sample["attack_cat"].value_counts())
print("[OK] Metadatas por clase en:", META_DIR)

# ===================== 4) Validación rápida del SYN (sin cargar entero) =====================
from collections import Counter
cnt_syn_attack = Counter()
cnt_syn_label  = Counter()

syn_reader = pd.read_csv(SYN_CSV, chunksize=200_000, low_memory=False)
for ch in tqdm(syn_reader, desc="Validando SYN (chunks)", unit="chunk"):
    cnt_syn_attack.update(ch["attack_cat"].astype(str).str.strip().tolist())
    cnt_syn_label.update(ch["label"].astype(str).str.strip().tolist())

print("\n[SYN] label:", dict(cnt_syn_label))
print("[SYN] attack_cat top15:", dict(Counter(cnt_syn_attack).most_common(15)))

# checks duros
bad1 = 0
bad2 = 0
syn_reader2 = pd.read_csv(SYN_CSV, chunksize=200_000, low_memory=False)
for ch in syn_reader2:
    ac = ch["attack_cat"].astype(str).str.lower()
    lb = pd.to_numeric(ch["label"], errors="coerce").fillna(0).astype(int)
    bad1 += int(((ac == "normal") & (lb != 0)).sum())
    bad2 += int(((ac != "normal") & (lb != 1)).sum())
print("\n[CHECK] Normal con label!=0:", bad1)
print("[CHECK] Ataque con label!=1:", bad2)
print("\n[FIN] Generación por clase lista (condicional real).")

In [ ]:
print("[REAL] attack_cat:", pd.read_csv("/content/KITSUNE_full_clean_v2.csv", low_memory=False)["attack_cat"].value_counts().head(30))
print("[SYN]  attack_cat:", pd.read_csv("/content/synthetic_kitsune_ctgan_v2.csv", low_memory=False)["attack_cat"].value_counts().head(30))

In [ ]:
from google.colab import files
files.download("/content/KITSUNE_full_clean_v2.csv")
files.download("/content/KITSUNE_train_sample_v2.csv")
files.download("/content/synthetic_kitsune_ctgan_v2.csv")
#files.download("/content/Kitsune_metadata_gc.json")